In [ ]:
import json
from groq import Groq

# 1. Set up the connection
client = Groq(api_key="")

# 2. The System Prompt (Crucial for JSON Mode)
system_prompt = """
You are a strict data extraction robot. 
Read the user's email and output ONLY a valid JSON object.
You must use these exact three keys: "sender_name", "urgency_level" (1-10), and "summary".
"""

# 3. The messy, unstructured input data
user_email = "Yo this is Dave from accounting. The payroll server is literally on fire and we can't pay anyone tomorrow. Call me back in 5 minutes or I'm quitting."

print("Extracting data into JSON... hold tight.\n")

# 4. The API Call (Notice the new response_format line!)
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_email}
    ],
    response_format={"type": "json_object"}  # <--- THE MAGIC LINE
)

# 5. Get the raw text from the AI
raw_ai_text = response.choices[0].message.content

# 6. Convert the text into a real Python Dictionary
email_data = json.loads(raw_ai_text)

print("🤖 Raw AI Output:")
print(raw_ai_text)

print("\n----------------------------------")
print("🐍 Accessing the Python Dictionary:")
print(f"Name extracted: {email_data['sender_name']}")
print(f"Panic Level: {email_data['urgency_level']}/10")
print(f"Summary: {email_data['summary']}")

Extracting data into JSON... hold tight.

🤖 Raw AI Output:
{
  "sender_name": "Dave",
   "urgency_level": 10,
   "summary": "Payroll server on fire, unable to pay employees"
}

----------------------------------
🐍 Accessing the Python Dictionary:
Name extracted: Dave
Panic Level: 10/10
Summary: Payroll server on fire, unable to pay employees


In [2]:
import tiktoken

# 1. Load the standard token dictionary
encoder = tiktoken.get_encoding("cl100k_base")

# 2. Our test sentence
my_text = "Yo this is Dave from accounting. The payroll server is literally on fire."

# 3. Translate the English into AI Tokens
tokens = encoder.encode(my_text)

print(f"Original Text: '{my_text}'")
print(f"Word Count: {len(my_text.split())}")
print(f"Token Count: {len(tokens)}")
print(f"What the AI actually sees: {tokens}")

Original Text: 'Yo this is Dave from accounting. The payroll server is literally on fire.'
Word Count: 13
Token Count: 15
What the AI actually sees: [65825, 420, 374, 20851, 505, 24043, 13, 578, 46208, 3622, 374, 16280, 389, 4027, 13]


In [2]:
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine

print("Downloading the AI Embedding Model (this takes a few seconds)...")
# We are downloading a tiny, ultra-fast embedding model from HuggingFace
model = SentenceTransformer('all-MiniLM-L6-v2')

# Our three test sentences
sentence1 = "My computer won't turn on."
sentence2 = "The laptop screen is completely black."
sentence3 = "I want to eat a pepperoni pizza."

print("Converting sentences into Math (Embeddings)...\n")
# 1. Turn sentences into vectors (lists of coordinates)
emb1 = model.encode(sentence1)
emb2 = model.encode(sentence2)
emb3 = model.encode(sentence3)

# 2. Calculate the distance between them (Cosine Similarity)
# 1.0 means exactly the same meaning, 0.0 means completely unrelated
sim_1_and_2 = 1 - cosine(emb1, emb2)
sim_1_and_3 = 1 - cosine(emb1, emb3)

print("🔍 AI SEMANTIC ANALYSIS:")
print(f"Similarity between PC issue and Laptop issue: {sim_1_and_2:.2f}")
print(f"Similarity between PC issue and Pizza: {sim_1_and_3:.2f}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4786.43it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Converting sentences into Math (Embeddings)...

🔍 AI SEMANTIC ANALYSIS:
Similarity between PC issue and Laptop issue: 0.36
Similarity between PC issue and Pizza: 0.14


In [1]:
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine
from groq import Groq

# 1. Setup our Tools
embedder = SentenceTransformer('all-MiniLM-L6-v2')
llm_client = Groq(api_key="")

# 2. Our "Private Database" (Information Llama 3 has NEVER seen)
database = [
    "The wifi password for the guest network is 'ILoveCats2026'.",
    "The company CEO is named Sarah Jenkins.",
    "Employees get 15 days of paid time off per year.",
    "The payroll server crashed on March 25th due to a spilled coffee."
]

# 3. Embed the Database (Convert text to Math Vectors)
print("1️⃣ Translating the database into math...")
db_embeddings = [embedder.encode(doc) for doc in database]

# 4. The User's Question
user_query = "Who is the boss of this company?"
print(f"\nUser asks: '{user_query}'")

# --- STEP 1: RETRIEVAL ---
print("2️⃣ Searching the database using Embeddings...")
query_embedding = embedder.encode(user_query)

best_score = -1
best_doc = ""

# Loop through our database to find the closest mathematical match
for i, doc_emb in enumerate(db_embeddings):
    similarity = 1 - cosine(query_embedding, doc_emb) # 1.0 is a perfect match
    if similarity > best_score:
        best_score = similarity
        best_doc = database[i]

print(f"✅ Found the right document! (Match Score: {best_score:.2f})")
print(f"📄 Document Text: '{best_doc}'")


# --- STEP 2: AUGMENTATION ---
print("\n3️⃣ Secretly stuffing the document into the AI's prompt...")
system_prompt = f"""
You are a helpful company assistant. Answer the user's question using ONLY the provided context.
If the answer is not in the context, say "I don't know." Do not make things up.

CONTEXT:
{best_doc}
"""

# --- STEP 3: GENERATION ---
print("4️⃣ Sending to Groq for generation...\n")
response = llm_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query}
    ]
)

print("🤖 FINAL AI RESPONSE:")
print(response.choices[0].message.content)

c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to

1️⃣ Translating the database into math...

User asks: 'Who is the boss of this company?'
2️⃣ Searching the database using Embeddings...
✅ Found the right document! (Match Score: 0.57)
📄 Document Text: 'The company CEO is named Sarah Jenkins.'

3️⃣ Secretly stuffing the document into the AI's prompt...
4️⃣ Sending to Groq for generation...

🤖 FINAL AI RESPONSE:
The CEO of the company is Sarah Jenkins.


In [6]:
import os

# 1. The path to your new file
file_path = "company_handbook.txt"

# 2. Open and read the real file from your Lenovo
print(f"Reading file: {file_path}...")
with open(file_path, "r", encoding="utf-16") as file:
    full_text = file.read()

# 3. The "Chunking" Logic
# We are going to split the document by double-newlines (paragraphs)
# In the real world, we use advanced libraries for this, but this is the raw logic.
paragraphs = full_text.split('\n\n')

# Clean up empty spaces
chunks = [p.strip() for p in paragraphs if p.strip()]

print(f"✅ Successfully chopped the document into {len(chunks)} chunks!")
print("\nHere is Chunk #1:")
print(f"--- {chunks[0]} ---")

Reading file: company_handbook.txt...
✅ Successfully chopped the document into 1 chunks!

Here is Chunk #1:
--- "The company was founded in 1994 by a guy named Steve. The breakroom fridge is cleaned on Fridays. The master server password is 'Swordfish'. ---


In [7]:
import os
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine
from groq import Groq

# 1. Setup our Tools (Using your NEW secret key)
embedder = SentenceTransformer('all-MiniLM-L6-v2')
llm_client = Groq(api_key="")

# 2. Read and Chunk the Local File (Your new superpower)
file_path = "company_handbook.txt"
print(f"📖 Reading {file_path}...")

try:
    with open(file_path, "r", encoding="utf-8") as file:
        full_text = file.read()
except UnicodeDecodeError:
    with open(file_path, "r", encoding="utf-16") as file:
        full_text = file.read()

# Chop into paragraphs (chunks)
chunks = [p.strip() for p in full_text.split('\n\n') if p.strip()]
print(f"✂️ Chopped into {len(chunks)} chunks.")

# 3. Embed the Chunks (Convert your file into Math)
print("🧮 Translating your file into a Vector Database...")
chunk_embeddings = [embedder.encode(doc) for doc in chunks]

# 4. The User's Question (Change this to ask about YOUR secrets!)
user_query = "What is the master server password?"
print(f"\n👤 User asks: '{user_query}'")

# --- RETRIEVAL ---
print("🔍 Searching your local file for the answer...")
query_embedding = embedder.encode(user_query)

best_score = -1
best_chunk = ""

for i, doc_emb in enumerate(chunk_embeddings):
    similarity = 1 - cosine(query_embedding, doc_emb)
    if similarity > best_score:
        best_score = similarity
        best_chunk = chunks[i]

# --- AUGMENTATION & GENERATION ---
print("🧠 Sending the exact right paragraph to Llama 3...\n")
system_prompt = f"""
You are a helpful company assistant. Answer the user's question using ONLY the provided context.
If the answer is not in the context, say "I don't know." Do not make things up.

CONTEXT:
{best_chunk}
"""

response = llm_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query}
    ]
)

print("🤖 FINAL AI RESPONSE:")
print(response.choices[0].message.content)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4763.89it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📖 Reading company_handbook.txt...
✂️ Chopped into 1 chunks.
🧮 Translating your file into a Vector Database...

👤 User asks: 'What is the master server password?'
🔍 Searching your local file for the answer...
🧠 Sending the exact right paragraph to Llama 3...

🤖 FINAL AI RESPONSE:
The master server password is 'Swordfish'.


In [9]:
import os
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine
from groq import Groq

# ==========================================
# PART 1: THE SETUP (This only happens once)
# ==========================================
print("⚙️ Booting up the AI Engine...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
llm_client = Groq(api_key="")

file_path = "company_handbook.txt"

# Read the file
try:
    with open(file_path, "r", encoding="utf-8") as file:
        full_text = file.read()
except UnicodeDecodeError:
    with open(file_path, "r", encoding="utf-16") as file:
        full_text = file.read()

# Chop and Embed
chunks = [p.strip() for p in full_text.split('\n\n') if p.strip()]
print(f"📚 Reading {len(chunks)} paragraphs into memory...")
chunk_embeddings = [embedder.encode(doc) for doc in chunks]

print("✅ System Ready! Type 'exit' or 'quit' to stop.")
print("-" * 50)

# ==========================================
# PART 2: THE CHAT LOOP (Runs forever)
# ==========================================
while True:
    # 1. Wait for the user to type something
    user_query = input("\n👤 You: ")
    
    # 2. Give the user a way to escape the endless loop
    if user_query.lower() in ['exit', 'quit']:
        print("👋 Shutting down the AI. See ya!")
        break
        
    # 3. Skip if the user just hits Enter by accident
    if not user_query.strip():
        continue

    # --- RETRIEVAL ---
    query_embedding = embedder.encode(user_query)
    best_score = -1
    best_chunk = ""

    for i, doc_emb in enumerate(chunk_embeddings):
        similarity = 1 - cosine(query_embedding, doc_emb)
        if similarity > best_score:
            best_score = similarity
            best_chunk = chunks[i]

    # --- GENERATION ---
    system_prompt = f"""
    You are a helpful company assistant. Answer the user's question using ONLY the provided context.
    If the answer is not in the context, say "I don't know." Do not make things up.

    CONTEXT:
    {best_chunk}
    """

    response = llm_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ]
    )

    # Print the answer and loop back to the top!
    print(f"🤖 AI: {response.choices[0].message.content}")
    

⚙️ Booting up the AI Engine...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4775.74it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📚 Reading 1 paragraphs into memory...
✅ System Ready! Type 'exit' or 'quit' to stop.
--------------------------------------------------
🤖 AI: Hello. Is there something I can help you with?
🤖 AI: I don't know.
🤖 AI: I don't know because it's not in the provided context.
🤖 AI: Is there something I can help you with?
👋 Shutting down the AI. See ya!


In [10]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Setup
embedder = SentenceTransformer('all-MiniLM-L6-v2')
file_path = "company_handbook.txt"

# 2. Read and Chunk
try:
    with open(file_path, "r", encoding="utf-8") as file:
        chunks = [p.strip() for p in file.read().split('\n\n') if p.strip()]
except:
    with open(file_path, "r", encoding="utf-16") as file:
        chunks = [p.strip() for p in file.read().split('\n\n') if p.strip()]

# 3. Create the Math (Embeddings)
print("🧠 Calculating math for the handbook...")
embeddings = embedder.encode(chunks)
embeddings = np.array(embeddings).astype('float32')

# 4. Create the FAISS Index (The Memory File)
dimension = embeddings.shape[1] # This is 384 for this model
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# 5. SAVE EVERYTHING TO DISK
faiss.write_index(index, "company_vault.index")
# We also need to save the text chunks so we can read them later
with open("chunks.txt", "w", encoding="utf-8") as f:
    for chunk in chunks:
        f.write(chunk + "---CHUNK_SPLIT---")

print("✅ DONE! Your 'company_vault.index' is saved on your Lenovo.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7057.31it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🧠 Calculating math for the handbook...
✅ DONE! Your 'company_vault.index' is saved on your Lenovo.


In [12]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from groq import Groq

# 1. LOAD THE MEMORY (Instant)
print("🚀 Loading the Vector Vault...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
llm_client = Groq(api_key="")

# Load the Math Index
index = faiss.read_index("company_vault.index")

# Load the Text Chunks (So the AI can read them back)
with open("chunks.txt", "r", encoding="utf-8") as f:
    chunks = f.read().split("---CHUNK_SPLIT---")
    chunks = [c for c in chunks if c.strip()] # Clean up

print(f"✅ Database loaded with {len(chunks)} paragraphs. Ask away!")

# 2. THE CHAT LOOP
while True:
    user_query = input("\n👤 Query: ")
    if user_query.lower() in ['exit', 'quit']: break

    # --- SEARCH (The FAISS Way) ---
    # Convert query to math
    query_emb = embedder.encode([user_query]).astype('float32')
    
    # Search the index for the Top 1 closest match
    # 'D' is distance (lower is better), 'I' is the Index (ID) of the chunk
    D, I = index.search(query_emb, k=1)
    best_chunk = chunks[I[0][0]]

    # --- GENERATION ---
    prompt = f"Use this context to answer: {best_chunk}\n\nQuestion: {user_query}"
    
    response = llm_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )

    print(f"🤖 AI: {response.choices[0].message.content}")

🚀 Loading the Vector Vault...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3828.55it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Database loaded with 1 paragraphs. Ask away!
🤖 AI: I'm just a language model, I don't have feelings or physical presence, so I don't have a personal state of being. However, I'm functioning properly and ready to assist you with any questions or tasks. How can I help you today?
🤖 AI: The text doesn't mention a CEO, but it does mention that the company was founded by a guy named Steve. It's possible that Steve is the CEO, but it's not explicitly stated.


In [1]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

print("🔄 Reading the updated handbook...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Read your updated file
try:
    with open("company_handbook.txt", "r", encoding="utf-8") as file:
        chunks = [p.strip() for p in file.read().split('\n\n') if p.strip()]
except UnicodeDecodeError:
    with open("company_handbook.txt", "r", encoding="utf-16") as file:
        chunks = [p.strip() for p in file.read().split('\n\n') if p.strip()]

print("🧠 Recalculating the math...")
embeddings = embedder.encode(chunks)
embeddings = np.array(embeddings).astype('float32')

# Overwrite the old memory
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)
faiss.write_index(index, "company_vault.index")

# Overwrite the text cache
with open("chunks.txt", "w", encoding="utf-8") as f:
    for chunk in chunks:
        f.write(chunk + "---CHUNK_SPLIT---")

print("✅ Memory Updated! The AI now knows your new secrets.")

c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔄 Reading the updated handbook...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7749.95it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🧠 Recalculating the math...
✅ Memory Updated! The AI now knows your new secrets.
